# M2: CLV 수준·구성·가격 좌표 임베딩 (Dunnhumby, seed 42)

사용자 보조표현을 총 CLV 수준·N/V 구성차 2차원과 V 기반 가격성향 1차원으로 고정합니다. 상품은 ID 기반 협업관계 2차원과 전체·카테고리 내 가격의 양의 혼합 1차원으로 대응합니다. 가격 예산 `beta=0.25`, 전체 보조강도 `rho=0.05`를 고정하고 하나의 67차원 이진 LightGCN·BPR·optimizer로 공동학습합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/clv-m2-lightgcn-runner
!git clone --branch feat/m2-joint-nv-lightgcn https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout f2793b1
!git rev-parse HEAD

In [ ]:
import json
import torch
from lightgcn_clv_constrained_economic_embedding import (
    configure_constrained_economic_run,
    preflight_summary,
    run_constrained_economic_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_constrained_economic_run()
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_constrained_economic_screen(cfg)

In [ ]:
from IPython.display import display
import pandas as pd

print('1) 절대지표: M1, rho=0, full, ID-only, 관계-only, 가격-only')
display(result_df)
print('2) 대조군별 전체 성과 비교')
display(pd.DataFrame(result_df.attrs['comparison']))
print('3) CLV 구간별 Top-10 변경')
display(pd.DataFrame(result_df.attrs['top10_overlap']))
print('4) 실제 점수 영향력')
display(pd.DataFrame(result_df.attrs['score_diagnostics']))
print('5) 사전 판정 규칙 결과')
print(json.dumps(result_df.attrs['screening_reading'], ensure_ascii=False, indent=2))
print('6) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))